In [1]:
import pyterrier as pt



In [2]:
!cp -r /root/nfs/jpq/experiments/ablation_amd_3090/msmarco-passage-train__tct_colbert__faiss2opq__M96_nbits8__ps159744__neg200__ibn__lr__ /tmp/

In [3]:
import pyterrier_dr
import pyterrier_dr.jpq

In [4]:
index = pyterrier_dr.jpq.JPQIndex("/tmp/msmarco-passage-train__tct_colbert__faiss2opq__M96_nbits8__ps159744__neg200__ibn__lr__")

In [5]:
model = pyterrier_dr.TctColBert()
model.model = model.model.from_pretrained("/tmp/msmarco-passage-train__tct_colbert__faiss2opq__M96_nbits8__ps159744__neg200__ibn__lr__").to(model.device)


In [26]:
import torch
device = torch.device("cuda")
codes = torch.from_numpy(index.codes.copy()).long().to(device)
docnos = index.docnos
meta = index.meta

In [34]:
meta

Metadata(type='dense_index', format='jpq', M=96, Ks=256, dsub=8, doc_count=8841823, opq=True)

In [35]:
M = meta.M
Dsub = meta.dsub
N = meta.doc_count

In [19]:
codebooks = torch.from_numpy(index.dvecs.copy()).to(device)

In [61]:
import torch, pandas as pd
#codes: torch.LongTensor  # [N, M] 
# dtype = int64 or int32 (torch.gather requirement)
#codebooks: torch.FloatTensor  # [M, Ks, Dsub]

opq = index.opq
if opq is not None:
    opq = torch.from_numpy(opq.copy()).to(device)

K=1000

def _retrieve(q: torch.FloatTensor):  # [D] or [B, D])
    if opq is not None:
        q = q @ opq
    
    q_sub = q.view(M, Dsub)
    LUT = torch.einsum('md,mkd->mk', q_sub, codebooks)
    LUT_exp = LUT.unsqueeze(0)

    # Gather: [N, M]
    scores = torch.gather(LUT_exp.expand(N, -1, -1), 2, codes.unsqueeze(-1)).squeeze(-1)
    
    # Final score: [N]
    scores = scores.sum(dim=1)

    indices = torch.topk(scores, K).indices
    return indices, scores[indices]

def _transformer(query_df : pd.DataFrame) -> pd.DataFrame:
    q_vecT = torch.from_numpy(query_df.iloc[0].query_vec).to(device)
    docidsT, scoresT = _retrieve(q_vecT)
    docids = docidsT.cpu().numpy()
    return pd.DataFrame({
            'qid' : query_df.iloc[0].qid,
            'score': scoresT.cpu().numpy(),
            'docno' : docnos[docids]})
    

In [62]:
gpupq = pt.apply.by_query(_transformer)

In [63]:
pipe = model >> gpupq

In [64]:
pipe.search("what are chemical reactions").head(2)

,qid,score,docno,rank
0,1,82.485085,1525431,0
1,1,82.452370,8572191,1


In [54]:
(model >> index.retriever_pq()).search("what are chemical reactions").head(2)

,qid,docid,docno,score,rank,query
0,1,1525431,1525431,82.485085,1,what are chemical reactions
1,1,8572191,8572191,82.452362,2,what are chemical reactions


In [67]:
from pyterrier.measures import*
pt.Experiment(
    [pipe],
    pt.get_dataset('msmarco_passage').get_topics('test-2019'),
    pt.get_dataset('msmarco_passage').get_qrels('test-2019'),
    [nDCG@10, "mrt", Recall@100, Recall@10, NumQ], batch_size=1, verbose=True, filter_by_qrels=True
)

/opt/miniconda3/envs/py312_jpq_dr/lib/python3.12/site-packages/pyterrier/_evaluation/_validation.py:41: UserWarning: Transformer (TctColBert('castorini/tct_colbert-msmarco') >> pt.apply.by_query()) at position 0 failed to validate: Cannot determine outputs for (TctColBert('castorini/tct_colbert-msmarco') >> pt.apply.by_query()) with inputs: ['qid', 'query'] - if your pipeline works, set validate='ignore' to remove this warning, or add transform_output method to the transformers in this pipeline to clarify how it works
  warn(


pt.Experiment:   0%|          | 0/43 [00:00<?, ?batches/s]

,name,NumQ,R@10,R@100,nDCG@10,mrt
0,(TctColBert('castorini/tct_colbert-msmarco') >...,43.0,0.150013,0.473867,0.669942,1081.62044


In [68]:
from pyterrier.measures import*
pt.Experiment(
    [pipe],
    pt.get_dataset('irds:msmarco-passage/dev/small').get_topics(),
    pt.get_dataset('irds:msmarco-passage/dev/small').get_qrels(),
    [RR@10,  "mrt", NumQ], batch_size=1, verbose=True, filter_by_qrels=True
)

/opt/miniconda3/envs/py312_jpq_dr/lib/python3.12/site-packages/pyterrier/_evaluation/_validation.py:41: UserWarning: Transformer (TctColBert('castorini/tct_colbert-msmarco') >> pt.apply.by_query()) at position 0 failed to validate: Cannot determine outputs for (TctColBert('castorini/tct_colbert-msmarco') >> pt.apply.by_query()) with inputs: ['qid', 'query'] - if your pipeline works, set validate='ignore' to remove this warning, or add transform_output method to the transformers in this pipeline to clarify how it works
  warn(


pt.Experiment:   0%|          | 0/6980 [00:00<?, ?batches/s]

,name,NumQ,RR@10,mrt
0,(TctColBert('castorini/tct_colbert-msmarco') >...,6980.0,0.323556,176273.246216


In [47]:
from pyterrier.measures import*
pt.Experiment(
    [model >> index.retriever_pq()],
    pt.get_dataset('msmarco_passage').get_topics('test-2019'),
    pt.get_dataset('msmarco_passage').get_qrels('test-2019'),
    [nDCG@10, "mrt", Recall@100, Recall@10], batch_size=1, verbose=True, filter_by_qrels=True
)

/opt/miniconda3/envs/py312_jpq_dr/lib/python3.12/site-packages/pyterrier/_evaluation/_validation.py:41: UserWarning: Transformer (TctColBert('castorini/tct_colbert-msmarco') >> JPQ-PQ) at position 0 failed to validate: Cannot determine outputs for (TctColBert('castorini/tct_colbert-msmarco') >> JPQ-PQ) with inputs: ['qid', 'query'] - if your pipeline works, set validate='ignore' to remove this warning, or add transform_output method to the transformers in this pipeline to clarify how it works
  warn(


pt.Experiment:   0%|          | 0/43 [00:00<?, ?batches/s]

,name,R@10,R@100,nDCG@10,mrt
0,(TctColBert('castorini/tct_colbert-msmarco') >...,0.150013,0.473867,0.669942,18778.010034


In [10]:
x = model >> index.retriever_pq()

In [11]:
x.search("what are chemical reactions?")

,qid,docid,docno,score,rank,query
0,1,1525431,1525431,82.839798,1,what are chemical reactions?
1,1,8572191,8572191,82.448654,2,what are chemical reactions?
2,1,758284,758284,82.042908,3,what are chemical reactions?
3,1,129901,129901,82.028717,4,what are chemical reactions?
4,1,203234,203234,81.987961,5,what are chemical reactions?
...,...,...,...,...,...,...
995,1,4257721,4257721,77.373024,996,what are chemical reactions?
996,1,5839900,5839900,77.372841,997,what are chemical reactions?
997,1,4785210,4785210,77.371651,998,what are chemical reactions?
998,1,1625391,1625391,77.371483,999,what are chemical reactions?


In [28]:
import numpy as np

x[1]._index

<faiss.swigfaiss_avx512.IndexPQ; proxy of <Swig Object of type 'faiss::IndexPQ *' at 0x7f1e2faccd50> >

In [45]:
%%timeit
vec = model.search("what are chemical reactions").iloc[0].query_vec

6.49 ms ± 142 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [46]:
%%timeit
r = x[1]._index.search(np.expand_dims(vec, 0), 1000)

380 ms ± 3.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [37]:
df = model.search("what are chemical reactions")

In [38]:
%%timeit
r = x[1].transform(df)

432 ms ± 6.91 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
